In [ ]:
from collections import defaultdict
import json
import re

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from tqdm.notebook import tqdm

Reading data

In [ ]:
with open("drive/MyDrive/dev_dpo.jsonl.FrancescoPeriti-LlamaDictionary-en_ML38BI.txt", "r") as ours:
    ours_pred = ours.read()

In [ ]:
to_split = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a lexicographer familiar with providing concise definitions of word meanings.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"

In [ ]:
ours_instances = ours_pred.split(to_split)

In [ ]:
target_pat = re.compile(f"Please provide a concise definition for the meaning of the word \"(.+)\" in the following sentence:")

In [ ]:
results_gold = defaultdict(list)
with open("drive/MyDrive/dev_dpo.jsonl", "r") as f:
    for line in f:
        line = json.loads(line)
        results_gold[line["target"]].append(line["definition"])

In [ ]:
results_our = defaultdict(list)
to_split_prompt = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
for i, instance_our in enumerate(ours_instances[1:]):
    prompt_our, pred_our = instance_our.split(to_split_prompt)
    target_our = re.match(target_pat, prompt_our).group(1)
    results_our[target_our].append(pred_our)

In [ ]:
!pip install -q FlagEmbedding

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3',
                       use_fp16=False,
                       devices="cuda")

Hyperparameter search (also for this model, to make the comparison fair - that both models are used with their best parameters)

In [ ]:
for linkage in ("complete", "average", "single"):
    print(linkage)
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        print(threshold)
        true = 0
        for target, definitions in tqdm(results_our.items()):

            definitions = list(set(definitions))
            if len(definitions) > 1:
                # print(len(definitions))
                clustering = AgglomerativeClustering(
                    n_clusters=None, metric="cosine", linkage=linkage, distance_threshold=threshold, compute_full_tree=True,
                )
                embeddings = model.encode(definitions,
                                        max_length=2048, # If you don't need such a long length, you can set a smaller value to speed up the encoding process.
                                        )['dense_vecs']
                labels = clustering.fit_predict(embeddings)

                if len(set(results_gold[target])) == len(set(labels)):
                    true += 1
            else:
                if len(definitions) == len(set(results_gold[target])):
                    true += 1
        print(true/len(results_our))
        print("-------------------")
    print("=========================================================")

Final prediction on EU data. It seems to me that differences between linkage types are not significant, but out of scope to check it now. At least we observe no dramatical degradation

> Add blockquote



In [ ]:
green = pd.read_csv("drive/MyDrive/Colab Notebooks/green-periti.tsv", sep="\t")

In [ ]:
green.head()

In [ ]:
green.shape

In [ ]:
len(green)

In [ ]:
to_concat = pd.DataFrame()
for target, target_df in tqdm(green.groupby("target")):
    definitions = target_df.prediction.unique()

    if len(definitions) > 1:
        # print(len(definitions))
        clustering = AgglomerativeClustering(
            n_clusters=None, metric="cosine", linkage="complete", distance_threshold=0.2, compute_full_tree=True,
        )
        embeddings = model.encode(definitions.tolist(),
                                max_length=2048, # If you don't need such a long length, you can set a smaller value to speed up the encoding process.
                                )['dense_vecs']
        labels = clustering.fit_predict(embeddings)
        labels_list = []
        for i, row in target_df.iterrows():
            idx = np.where(definitions == row.prediction)
            labels_list.append(labels[idx].tolist()[0])
        target_df["labels"] = labels_list
    else:
        target_df["labels"] = [0 for i in range(len(target_df))]
    to_concat = pd.concat([to_concat, target_df])
assert len(to_concat) == len(green)
to_concat = to_concat.reset_index(drop=True)
to_concat.to_csv("drive/MyDrive/green-periti.clustered.tsv", sep='\t', index=False)
to_concat.head()